# Setup IBGE — Carga de Municípios

**Notebook utilitário — rodar UMA VEZ antes de executar o pipeline
principal pela primeira vez, ou quando os dados do IBGE precisarem
ser atualizados.**

Este notebook não faz parte do pipeline principal (Bronze → Silver
→ Gold). Ele é responsável por buscar os dados de municípios
brasileiros diretamente da API pública do IBGE e salvar o resultado
no container `raw` do Data Lake, onde o notebook
`03_gold_physical_lojas` vai consumir para o enriquecimento de
população por cidade (KPI 7).

**O que este notebook faz:**
- Chama a API pública do IBGE (`servicodados.ibge.gov.br`) para
  obter a lista de todos os municípios brasileiros com nome, UF
  e código IBGE.
- Chama a API SIDRA do IBGE para obter a população de cada
  município (Censo 2022).
- Consolida os dados em um DataFrame e salva como CSV no container
  `squad3/bronze/ibge_municipios` (container de escrita da squad,
  não no raw que é somente leitura).

**Pré-requisito:** o domínio `servicodados.ibge.gov.br` precisa
estar acessível a partir do cluster Databricks. Se o ambiente
restringir acesso HTTP externo, use a Opção 3 (CSV estático)
descrita no README do projeto.

**Frequência de atualização sugerida:** anual, após publicação de
novo Censo ou estimativa populacional pelo IBGE.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

In [0]:
import requests
import pandas as pd
from io import BytesIO


####Checagem automática de atualidade (elimina o passo manual)

Antes deste notebook rodava só quando alguém lembrava. Agora ele é chamado automaticamente pelo gold_physical_lojas (via %run) em toda execução do pipeline, mas só efetivamente busca dados novos na API do IBGE se os dados atuais tiverem mais de 365 dias (população muda pouco ano a ano — não faz sentido chamar a API toda vez).

Se os dados já estiverem atualizados, o notebook imprime um aviso e encerra cedo com dbutils.notebook.exit() — sem custo de rede, sem passo manual, e sem risco de esquecer de rodar.

Bug corrigido: a primeira versão tentava usar dbutils.notebook.exit() para encerrar cedo quando os dados já estavam atualizados — mas isso só funciona com dbutils.notebook.run(), não dentro de um %run (que é como este notebook é chamado). Agora usa uma flag (ibge_precisa_atualizar) e todas as células de busca/gravação abaixo rodam dentro de um if, virando um no-op rápido quando os dados já estão em dia — sem tentar interromper a execução do notebook no meio.



In [0]:
IBGE_BRONZE_PATH = f"{BRONZE_BASE_PATH}ibge_municipios"
IBGE_METADATA_PATH = f"{IBGE_BRONZE_PATH}_metadata"
LIMITE_DIAS_ATUALIDADE = 365

data_ultima_geracao = None
try:
    df_metadata = read_delta(IBGE_METADATA_PATH, adls_options)
    data_ultima_geracao = (
        df_metadata.orderBy(col("gerado_em").desc())
        .select("gerado_em").first()["gerado_em"]
    )
    # Bug corrigido: o timestamp que volta do Delta via .first() e
    # "naive" (sem timezone), enquanto datetime.now(timezone.utc) e
    # "aware" -- Python nao deixa subtrair os dois tipos
    # (TypeError: can't subtract offset-naive and offset-aware
    # datetimes). Forca timezone UTC explicitamente se estiver ausente.
    if data_ultima_geracao is not None and data_ultima_geracao.tzinfo is None:
        data_ultima_geracao = data_ultima_geracao.replace(tzinfo=timezone.utc)
except Exception:
    data_ultima_geracao = None

# Bug corrigido: dbutils.notebook.exit() só funciona quando o notebook
# é chamado via dbutils.notebook.run() -- dentro de um %run (como este
# notebook e chamado pelo gold_physical_lojas), ele NAO encerra
# graciosamente, gera erro ("execution did not finish successfully").
# Em vez de tentar sair do notebook no meio, usamos uma flag e
# envolvemos as celulas custosas (chamadas de API) num 'if' abaixo.
ibge_precisa_atualizar = True

if data_ultima_geracao is not None:
    dias_desde_geracao = (datetime.now(timezone.utc) - data_ultima_geracao).days
    print(f"[Frente D] Última geração do IBGE: {data_ultima_geracao} "
          f"({dias_desde_geracao} dia(s) atrás).")

    if dias_desde_geracao < LIMITE_DIAS_ATUALIDADE:
        print(
            f"[OK] Dados do IBGE ainda dentro da validade "
            f"(< {LIMITE_DIAS_ATUALIDADE} dias). Nenhuma chamada à API "
            f"necessária — pulando as células de busca abaixo."
        )
        ibge_precisa_atualizar = False
    else:
        print(
            f"[ALERTA] Dados do IBGE com {dias_desde_geracao} dia(s) — "
            f"acima do limite de {LIMITE_DIAS_ATUALIDADE}. Buscando "
            f"dados atualizados na API..."
        )
else:
    print(
        "[AVISO] Nenhum metadado de geração anterior encontrado — "
        "executando carga inicial do IBGE."
    )

## Passo 1 — Testar conectividade com a API IBGE

Antes de fazer as chamadas completas, confirma que o domínio está
acessível a partir deste cluster. Se falhar aqui, o ambiente está
bloqueando acesso HTTP externo — consulte o tech lead sobre
liberação do domínio ou use o CSV estático como alternativa.

In [0]:
URL_TESTE = "https://servicodados.ibge.gov.br/api/v1/localidades/regioes"

try:
    resp = requests.get(URL_TESTE, timeout=10)
    resp.raise_for_status()
    print(f"[OK] Conectividade com a API IBGE confirmada. "
          f"Status: {resp.status_code}")
except Exception as e:
    raise RuntimeError(
        f"ERRO: não foi possível acessar a API IBGE. "
        f"Verifique se o domínio 'servicodados.ibge.gov.br' está "
        f"liberado no firewall do ambiente Databricks. "
        f"Detalhe: {e}"
    )

## Passo 2 — Buscar lista de municípios (nome + UF + código IBGE)

In [0]:
if ibge_precisa_atualizar:
    URL_MUNICIPIOS = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

    resp_municipios = requests.get(URL_MUNICIPIOS, timeout=30)
    resp_municipios.raise_for_status()

    dados_municipios = resp_municipios.json()

    df_municipios = pd.DataFrame([
        {
            "codigo_ibge": str(m["id"]),
            "nome_municipio": m["nome"].upper().strip(),
            "sigla_uf": (
                (m.get("microrregiao") or {})
                .get("mesorregiao", {})
                .get("UF", {})
                .get("sigla", "")
            ),
            "nome_uf": (
                (m.get("microrregiao") or {})
                .get("mesorregiao", {})
                .get("UF", {})
                .get("nome", "")
            ),
        }
        for m in dados_municipios
        if m.get("id") and m.get("nome")
    ])

    print(f"[OK] {len(df_municipios)} municípios carregados da API IBGE.")
    display(df_municipios.head(10))
else:
    print("[SKIP] Busca de municípios pulada — dados já atualizados.")

## Passo 3 — Buscar população por município (Censo 2022)

Usa a API de Agregados do IBGE (v3), tabela 9514 — população residente por município, Censo Demográfico 2022. A tabela 4714 (usada anteriormente) era do Censo 2010 e foi substituída pela 9514 para o Censo 2022.

In [0]:
if ibge_precisa_atualizar:
    # API de Agregados v3 — tabela 9514, variável 93 (pop. residente),
    # nível N6 (município), período 2022
    URL_POPULACAO = (
        "https://servicodados.ibge.gov.br/api/v3/agregados/9514"
        "/periodos/2022/variaveis/93?localidades=N6[all]"
    )

    try:
        resp_pop = requests.get(URL_POPULACAO, timeout=60)
        resp_pop.raise_for_status()
        dados_pop = resp_pop.json()

        # Estrutura da resposta v3:
        # [{"resultados": [{"series": [{"localidade": {"id": "...", "nome": "..."},
        #                               "serie": {"2022": "valor"}}]}]}]
        registros = []
        for resultado in dados_pop[0].get("resultados", []):
            for serie in resultado.get("series", []):
                codigo = serie.get("localidade", {}).get("id", "")
                valor  = serie.get("serie", {}).get("2022", None)
                if codigo and valor and valor not in ("-", "..", "..."):
                    registros.append({
                        "codigo_ibge": str(codigo),
                        "populacao": int(valor),
                    })

        df_populacao = pd.DataFrame(registros)
        print(f"[OK] População carregada para {len(df_populacao)} municípios (Censo 2022).")

    except Exception as e:
        print(
            f"[AVISO] Não foi possível carregar dados de população da API IBGE. "
            f"O enriquecimento de população ficará nulo. Detalhe: {e}"
        )
        df_populacao = pd.DataFrame(columns=["codigo_ibge", "populacao"])
else:
    print("[SKIP] Busca de população pulada — dados já atualizados.")


####Passo 3b — Buscar renda domiciliar per capita por município (Censo 2022)

Regra oficial (Squad 3 — Negócio 7): enriquecimento IBGE deve trazer população E renda. Usa a tabela Sidra 10295 — "Moradores em domicílios particulares permanentes... valor do rendimento domiciliar mensal per capita, médio e mediano", com nível de território incluindo município (MU).

Importante: ao contrário da tabela de população (9514, variável 93 fixa), a tabela 10295 tem várias variáveis (renda média, mediana) e classificações (sexo, cor/raça, faixa etária), cada uma com um id específico que pode mudar. Em vez de cravar um número que não pude confirmar com certeza, o código abaixo consulta os metadados da própria tabela em tempo de execução e descobre dinamicamente:

o id da variável de "valor do rendimento... médio" (não mediano);
o id da categoria "Total" em cada classificação (sexo, cor/raça, idade), para pegar o agregado geral e não uma quebra específica.
Se a estrutura da API mudar e a busca dinâmica não encontrar essas categorias, o notebook avisa claramente e segue sem dado de renda (nunca falha silenciosamente nem assume um id arriscado).

In [0]:
if ibge_precisa_atualizar:
    AGREGADO_RENDA = 10295
    renda_disponivel = False
    df_renda = None

    try:
        resp_meta = requests.get(
            f"https://servicodados.ibge.gov.br/api/v3/agregados/{AGREGADO_RENDA}/metadados",
            timeout=30,
        )
        resp_meta.raise_for_status()
        metadados_renda = resp_meta.json()

        # 1) Acha a variável de renda MÉDIA (evita pegar a mediana por engano)
        variavel_renda_id = None
        for v in metadados_renda.get("variaveis", []):
            nome_v = v["nome"].lower()
            if "rendimento" in nome_v and "médio" in nome_v and "mediano" not in nome_v:
                variavel_renda_id = v["id"]
                break

        # 2) Acha a categoria "Total" em cada classificação (sexo, cor/raça, idade)
        classificacoes_total = []
        for c in metadados_renda.get("classificacoes", []):
            cat_total = next(
                (cat["id"] for cat in c.get("categorias", []) if cat["nome"] == "Total"),
                None,
            )
            if cat_total is not None:
                classificacoes_total.append(f"{c['id']}[{cat_total}]")

        if variavel_renda_id is None or not classificacoes_total:
            print(
                "[AVISO] Não foi possível identificar automaticamente a "
                "variável/classificação de renda na tabela 10295. "
                "Enriquecimento de renda será ignorado nesta execução — "
                "revisar manualmente a estrutura da tabela em "
                "https://sidra.ibge.gov.br/tabela/10295."
            )
        else:
            classificacao_param = "|".join(classificacoes_total)
            URL_RENDA = (
                f"https://servicodados.ibge.gov.br/api/v3/agregados/{AGREGADO_RENDA}"
                f"/periodos/2022/variaveis/{variavel_renda_id}"
                f"?localidades=N6[all]&classificacao={classificacao_param}"
            )
            resp_renda = requests.get(URL_RENDA, timeout=60)
            resp_renda.raise_for_status()
            dados_renda = resp_renda.json()

            registros_renda = []
            for item in dados_renda:
                for resultado in item.get("resultados", []):
                    for serie in resultado.get("series", []):
                        codigo_localidade = serie["localidade"]["id"]
                        for _, valor in serie["serie"].items():
                            registros_renda.append({
                                "codigo_ibge": str(codigo_localidade),
                                "renda_media_per_capita": (
                                    None if valor in ("...", "-", None)
                                    else float(valor)
                                ),
                            })

            df_renda = pd.DataFrame(registros_renda)
            renda_disponivel = True
            print(f"[OK] Renda: {len(df_renda)} município(s) carregados "
                  f"(variável id={variavel_renda_id}).")

    except Exception as e:
        print(
            f"[AVISO] Falha ao buscar renda por município (tabela 10295). "
            f"Enriquecimento de renda será ignorado nesta execução. Erro: {e}"
        )
else:
    print("[SKIP] Busca de renda pulada — dados já atualizados.")

## Passo 4 — Consolidar e salvar no container raw

In [0]:
if ibge_precisa_atualizar:
    df_ibge_final = df_municipios.merge(
        df_populacao,
        on="codigo_ibge",
        how="left",
    )

    if renda_disponivel and df_renda is not None:
        df_ibge_final = df_ibge_final.merge(df_renda, on="codigo_ibge", how="left")
    else:
        df_ibge_final["renda_media_per_capita"] = None

    print(f"[OK] Dataset consolidado: {len(df_ibge_final)} municípios.")
    print(f"     Com população: {df_ibge_final['populacao'].notna().sum()}")
    print(f"     Sem população: {df_ibge_final['populacao'].isna().sum()}")
    print(f"     Com renda: {df_ibge_final['renda_media_per_capita'].notna().sum()}")
    print(f"     Sem renda: {df_ibge_final['renda_media_per_capita'].isna().sum()}")

    # Usa print em vez de display() para evitar travamento do Serverless
    # ao renderizar tabelas pandas grandes
    print("\nAmostra (5 primeiras linhas):")
    print(df_ibge_final.head(5).to_string(index=False))
else:
    print("[SKIP] Consolidação pulada — dados já atualizados, nada para salvar.")

## Passo 5 — Salvar como CSV no container squad3 (bronze)

O arquivo do IBGE é um dado de enriquecimento **gerado pelo pipeline**,
não uma fonte original externa — por isso vai para `squad3/bronze/`
(container de escrita da squad), não para `raw` (somente leitura).

In [0]:
if ibge_precisa_atualizar:
    # Converte para Spark e salva diretamente — sem coalesce(1) para
    # evitar gargalo de coleta no driver com ~5.570 linhas
    df_ibge_spark = spark.createDataFrame(df_ibge_final.fillna("").astype(str))

    (
        df_ibge_spark
        .write
        .format("csv")
        .option("header", "true")
        .option("encoding", "UTF-8")
        .options(**adls_options)
        .mode("overwrite")
        .save(IBGE_BRONZE_PATH)
    )

    # Frente D: grava o metadado de geração, usado pela checagem de
    # atualidade automática na próxima execução do pipeline.
    df_metadata_novo = spark.createDataFrame(
        [(datetime.now(timezone.utc),)], schema="gerado_em timestamp"
    )
    write_delta(df_metadata_novo, IBGE_METADATA_PATH, adls_options, mode="overwrite")

    print(f"[OK] Arquivo salvo em '{IBGE_BRONZE_PATH}'.")
    print(f"     Total de municípios: {df_ibge_final.shape[0]}")
    print(f"     Colunas: {list(df_ibge_final.columns)}")
    print(f"     Metadado de atualidade gravado em '{IBGE_METADATA_PATH}'.")
else:
    print(
        f"[SKIP] Gravação pulada — arquivo em '{IBGE_BRONZE_PATH}' "
        f"já está atualizado (dentro dos {LIMITE_DIAS_ATUALIDADE} dias)."
    )
